# Week 1 · Day 2 — GROUP BY, HAVING, Aggregates
**Goal:** Stop looking at individual rows. Start answering questions about groups.

**Schedule:**
- Block 1 · 9:00–10:30am · Concept + Drills
- Block 2 · 11:00am–12:00pm · Apply + AI Audit

**Day 1 gap to fix today:** Zoom out first — confirm there's a problem across groups before drilling into why.

---
## Business First Prompt

> *Your manager asks: "Which regions are driving the most revenue and which ones are lagging? I want to know where to focus our sales team."*

Write 3–5 sentences below before touching any data:
- What would you measure?
- What does "driving the most revenue" look like in the data?
- What would you need to compare to call a region "lagging"?

**Your answer:**

> I would measure total revenue per region over a quarter period. Driving the most revenue in the data looks like the sum of revenue per region. I would get the last quarter average of total revenue and compare to current output per region. The comparison would determine if a region is "lagging" in comparison to the comapny total.

---
## Setup — run this first

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('week1_ecommerce.db')
print('Connected to week1_ecommerce.db')

Connected to week1_ecommerce.db


---
## Table Preview — run this so you know what you're working with

In [5]:
for table in ['orders', 'customers', 'order_items']:
    print(f'\n--- {table} ---')
    df = pd.read_sql_query(f'SELECT * FROM {table} LIMIT 3', conn)
    print(f'Columns: {list(df.columns)}')
    display(df)


--- orders ---
Columns: ['order_id', 'customer_id', 'order_date', 'status', 'total_amount', 'region']


,order_id,customer_id,order_date,status,total_amount,region
0,1001,1,2024-01-05,completed,250.00,West
1,1002,2,2024-01-12,cancelled,89.99,East
2,1003,3,2024-01-20,completed,430.50,West



--- customers ---
Columns: ['customer_id', 'name', 'email', 'signup_date', 'segment', 'country']


,customer_id,name,email,signup_date,segment,country
0,1,Alice Martin,alice@email.com,2022-03-15,VIP,US
1,2,Bob Chen,bob@email.com,2022-07-22,Regular,CA
2,3,Sara Lopez,sara@email.com,2023-01-10,Regular,US



--- order_items ---
Columns: ['item_id', 'order_id', 'product_name', 'category', 'quantity', 'unit_price']


,item_id,order_id,product_name,category,quantity,unit_price
0,1,1001,Wireless Mouse,Electronics,1,45.0
1,2,1001,USB Hub,Electronics,2,25.0
2,3,1001,Desk Mat,Accessories,1,35.0


---
## Concept — Aggregate Functions

Instead of returning individual rows, these collapse many rows into one number.

| Function | What it does | Example |
|----------|-------------|----------|
| `COUNT(*)` | Count rows | How many orders? |
| `SUM(column)` | Add up values | Total revenue |
| `AVG(column)` | Average value | Average order size |
| `MAX(column)` | Highest value | Biggest order |
| `MIN(column)` | Lowest value | Smallest order |

```sql
SELECT COUNT(*) as total_orders,
       SUM(total_amount) as total_revenue,
       AVG(total_amount) as avg_order_value
FROM orders
WHERE status = 'completed'
```

In [6]:
# EXAMPLE — overall numbers across all orders
q = """
SELECT COUNT(*) as total_orders,
       SUM(total_amount) as total_revenue,
       AVG(total_amount) as avg_order_value,
       MAX(total_amount) as largest_order,
       MIN(total_amount) as smallest_order
FROM orders
WHERE status = 'completed'
"""

df = pd.read_sql_query(q, conn)
display(df)

,total_orders,total_revenue,avg_order_value,largest_order,smallest_order
0,19,8595.5,452.394737,980.0,145.0


---
## Concept — GROUP BY

Splits your data into groups and runs the aggregate function on each group separately.

Think of it as: *"Give me the SUM / COUNT / AVG — but broken out by each value in this column."*

```sql
SELECT column_to_group_by,
       COUNT(*) as count,
       SUM(amount) as total
FROM table_name
WHERE ...
GROUP BY column_to_group_by
ORDER BY total DESC
```

**Rule:** Any column in your SELECT that is NOT an aggregate must appear in GROUP BY.

In [4]:
# EXAMPLE — revenue broken out by region
# This is the zoom-out query you needed in Block 2 yesterday
q = """
SELECT region,
       COUNT(*) as total_orders,
       SUM(total_amount) as total_revenue,
       ROUND(AVG(total_amount), 2) as avg_order_value
FROM orders
WHERE status = 'completed'
GROUP BY region
ORDER BY total_revenue DESC
"""

df = pd.read_sql_query(q, conn)
display(df)

,region,total_orders,total_revenue,avg_order_value
0,West,7,3690.5,527.21
1,South,5,2015.0,403.00
2,East,3,1590.0,530.00
3,North,4,1300.0,325.00


In [ ]:
# EXAMPLE — order count by status (all statuses)
q = """
SELECT status,
       COUNT(*) as order_count,
       SUM(total_amount) as total_value
FROM orders
GROUP BY status
ORDER BY order_count DESC
"""

df = pd.read_sql_query(q, conn)
display(df)

---
## Concept — HAVING

HAVING filters groups — just like WHERE filters rows, but it runs AFTER the GROUP BY.

- Use `WHERE` to filter rows before grouping
- Use `HAVING` to filter groups after grouping

```sql
SELECT region, COUNT(*) as total_orders
FROM orders
GROUP BY region
HAVING COUNT(*) > 5
```

In [5]:
# EXAMPLE — only show regions with more than 2 completed orders
q = """
SELECT region,
       COUNT(*) as completed_orders,
       SUM(total_amount) as total_revenue
FROM orders
WHERE status = 'completed'
GROUP BY region
HAVING COUNT(*) > 2
ORDER BY total_revenue DESC
"""

df = pd.read_sql_query(q, conn)
display(df)

,region,completed_orders,total_revenue
0,West,7,3690.5
1,South,5,2015.0
2,East,3,1590.0
3,North,4,1300.0


---
## Clause Order — the full picture so far

```sql
SELECT    -- columns + aggregates
FROM      -- table
WHERE     -- filter rows (before grouping)
GROUP BY  -- split into groups
HAVING    -- filter groups (after grouping)
ORDER BY  -- sort
LIMIT     -- cap results
```

This order is fixed. Always.

---
## Drill 1 — Count orders by region

Show how many orders each region has — all statuses included.
Columns: region, order_count. Highest count first.

Clauses: `SELECT` · `FROM` · `GROUP BY` · `ORDER BY`

In [11]:
q1 = """SELECT region,  
        COUNT(*) as order_count
        FROM orders
        GROUP BY region
        ORDER BY order_count DESC
"""

df = pd.read_sql_query(q1, conn)
display(df)

,region,order_count
0,West,11
1,East,8
2,North,6
3,South,5


---
## Drill 2 — Total revenue by region (completed only)

Show total revenue per region for completed orders only.
Columns: region, total_revenue, avg_order_value. Highest revenue first.

Clauses: `SELECT` · `FROM` · `WHERE` · `GROUP BY` · `ORDER BY`

Note: wrap AVG in `ROUND(..., 2)` to keep it clean.

In [8]:
q2 = """Select region,
        SUM(total_amount) as total_revenue,
        AVG(total_amount) as avg_order_value
        FROM orders
        WHERE status = 'completed'
        GROUP BY region
        ORDER BY total_revenue DESC
"""

df = pd.read_sql_query(q2, conn)
display(df)

,region,total_revenue,avg_order_value
0,West,3690.5,527.214286
1,South,2015.0,403.000000
2,East,1590.0,530.000000
3,North,1300.0,325.000000


---
## Drill 3 — Failed order value by region

For cancelled and refunded orders — show total lost value per region.
Columns: region, failed_orders, lost_revenue. Highest lost revenue first.

Clauses: `SELECT` · `FROM` · `WHERE` · `GROUP BY` · `ORDER BY`

Hint: use `IN ('cancelled', 'refunded')` in your WHERE.

In [ ]:
q3 = """SELECT region,
        COUNT(*)  as failed_orders,
        SUM(total_amount) as lost_revenue
        FROM orders
        WHERE status IN ('cancelled', 'refunded')
        GROUP BY region
        ORDER BY lost_revenue DESC

"""

df = pd.read_sql_query(q3, conn)
display(df)

,region,failed_orders,lost_revenue
0,East,2,535.0
1,West,1,440.0
2,North,2,395.0


---
## Drill 4 — Regions with high average order value

Show only regions where the average completed order is above $300.
Columns: region, avg_order_value, total_orders.

Clauses: `SELECT` · `FROM` · `WHERE` · `GROUP BY` · `HAVING` · `ORDER BY`

Hint: this is where HAVING comes in — you're filtering a group by an aggregate.

In [17]:
q4 = """SELECT region,
        COUNT(*) as total_orders,
        AVG(total_amount) as avg_order_value
        FROM orders
        GROUP BY region
        HAVING AVG(total_amount) > 300
        ORDER BY total_orders DESC
"""

df = pd.read_sql_query(q4, conn)
display(df)

,region,total_orders,avg_order_value
0,West,11,417.318182
1,East,8,302.248750
2,South,5,403.000000


---
## Block 2 · 11:00am — Applied Challenge

> *Same manager, same question as yesterday: "We think the West region is underperforming. Show me what's going on."*

Today you have GROUP BY. Do it properly — zoom out first, then zoom in.

**Query 1:** Zoom out — compare all regions on completed revenue and order count so you can confirm whether West actually has a problem.

**Query 2:** Zoom in — break down West's orders by status to see where the value is being lost.

Write your framing sentence first.

**My approach — zoom out then zoom in:**

> I will first look at the broader picture, confirm if indeed the West region is underperforming. I will compare the west region to all others in whithin a 3 month period. According to my observations, I will pinpoint the root cause of the underperformance. I will analyze Item type and value and compare it to the order status.

In [18]:
for table in ['orders', 'customers', 'order_items']:
    print(f'\n--- {table} ---')
    df = pd.read_sql_query(f'SELECT * FROM {table} LIMIT 3', conn)
    print(f'Columns: {list(df.columns)}')
    display(df)


--- orders ---
Columns: ['order_id', 'customer_id', 'order_date', 'status', 'total_amount', 'region']


,order_id,customer_id,order_date,status,total_amount,region
0,1001,1,2024-01-05,completed,250.00,West
1,1002,2,2024-01-12,cancelled,89.99,East
2,1003,3,2024-01-20,completed,430.50,West



--- customers ---
Columns: ['customer_id', 'name', 'email', 'signup_date', 'segment', 'country']


,customer_id,name,email,signup_date,segment,country
0,1,Alice Martin,alice@email.com,2022-03-15,VIP,US
1,2,Bob Chen,bob@email.com,2022-07-22,Regular,CA
2,3,Sara Lopez,sara@email.com,2023-01-10,Regular,US



--- order_items ---
Columns: ['item_id', 'order_id', 'product_name', 'category', 'quantity', 'unit_price']


,item_id,order_id,product_name,category,quantity,unit_price
0,1,1001,Wireless Mouse,Electronics,1,45.0
1,2,1001,USB Hub,Electronics,2,25.0
2,3,1001,Desk Mat,Accessories,1,35.0


In [34]:
# Query 1 — zoom out: all regions compared
q5 = """SELECT region,

        COUNT(*) as total_orders,
        SUM(total_amount) as total_revenue,
        AVG(total_amount) as avg_order_value
        FROM orders
         WHERE order_date >= '2024-09-30' AND order_date <= '2024-12-31'
        GROUP BY region
         ORDER BY total_revenue DESC

"""

df = pd.read_sql_query(q5, conn)
display(df)

#NOTE: HAVING is only for aggregates and WHERE is for row filters. WHERE comes before GROUP BY and HAVING comes after.

,region,total_orders,total_revenue,avg_order_value
0,West,3,2340.0,780.000000
1,South,2,975.0,487.500000
2,North,2,625.0,312.500000
3,East,3,398.0,132.666667


In [12]:
# Query 2 — zoom in: West broken down by status
q6 = """SELECT order_id, order_date, status, total_amount
        FROM orders
        WHERE region = 'West' AND status IN ('cancelled', 'refunded', 'completed') AND order_date >= '2024-09-30' AND order_date <= '2024-12-31'
        GROUP BY order_id
        

"""

df = pd.read_sql_query(q6, conn)
display(df)

,order_id,order_date,status,total_amount
0,1023,2024-10-22,completed,720.0
1,1027,2024-12-02,completed,980.0
2,1030,2024-12-28,completed,640.0


In [50]:
# My own Query 3 - Give actual percentage comparison in revenue from WEST and EAST regions. With help from chatgpt.
q7 = """
SELECT 
    ROUND(
        (
            SUM(CASE WHEN region = 'West' THEN total_amount ELSE 0 END) -
            SUM(CASE WHEN region = 'East' THEN total_amount ELSE 0 END)
        ) * 100.0 /
        SUM(CASE WHEN region = 'East' THEN total_amount ELSE 0 END),
    2) AS pct_more_west_than_east
FROM orders
WHERE region IN ('West', 'East')
  AND order_date >= '2024-09-30'
  AND order_date <= '2024-12-31'
"""
df = pd.read_sql_query(q7, conn)
display(df)

,pct_more_west_than_east
0,487.94


**What does the data say now?**
Is West actually underperforming? What's the evidence? What would you tell your manager?

> West region is not underperforming. In comparison to other regions its actually leading the way in revenue with the same amount of orders. With the same amount of orders as the East region in the last quarter, West region 487.94% provided more revenue. The region to lookout for is the East, even west and south with 2 orders each have brought in more revenue. 

---
## Day 2 Checkpoint — answer before Day 3

Plain English — no SQL needed.

**1. What is the difference between WHERE and HAVING? Give a real example of when you'd use each.**

> WHERE is used to filter row values in a column and HAVING filters on agreggates after GROUP BY. For example Id use WHERE to give me results for only completed orders from the status column. I would use HAVING to filter further after grouping for example customer segments where total revenue across all their orders exceeds $500.

**2. Your manager asks: "How many orders did each customer segment place last quarter, and which segments placed more than 3 orders?" Which clauses do you need?**

> I would us SELECT, COUNT(*), FROM, WHERE, GROUP BY, HAVING.

**3. After running today's zoom-out query — is West actually underperforming? What does the data say compared to your gut feeling yesterday?**

> No in the end West was not underperforming. My inital observation was that there was not enough pointers to indicate that west was underperforming. Reviewing the data further, comparing West to the other regions showed that west was actually leading the way. I learned that its important to consider the broader picture to give context to findings.